# 3.2 — Les optimisateurs : de SGD à Adam, ce qui change vraiment

**Navigation** : [<< 3.1-Retropropagation](3.1-Retropropagation.ipynb) | [Index](README.md) | [3.3 — Régularisation >>](3.3-Regularisation.ipynb)

**Kernel** : .NET C# (MathNet, TorchSharp et témoin sklearn) · **CPU** : oui · **Durée** : 2–3 heures (étude et exercices)

***

## Introduction

[3.1](3.1-Retropropagation.ipynb) a entraîné un réseau avec la descente de gradient. Les variantes comme momentum et Adam ajoutent des états et modifient la règle de mise à jour ; ce carnet ouvre les boîtes centrales et les confronte à un moteur réel :

| optimiseur | année | idée en une ligne |
|---|---|---|
| SGD + momentum | 1964 / 1986 | une boule qui accumule de l'élan dans le ravin |
| Adagrad | 2011 | un learning rate **par paramètre**, qui décroît avec l'historique |
| RMSProp | 2012 | la moyenne glissante qui répare l'accumulation infinie d'Adagrad |
| Adam | 2014 | RMSProp **+** momentum, avec la correction de biais des premiers pas |

La discipline est celle de 3.1 : **from scratch PUIS framework**. Chaque optimiseur est écrit
en C# (l'état — momentum, accumulateurs — est explicite), puis **validé contre
`TorchSharp.optim`** : les paramètres sont comparés sur plusieurs pas avec une tolérance explicite. Ensuite seulement, le grand comparatif — trajectoires superposées sur la surface
de Beale, courbes de perte du MLP de 3.1 sur 3 graines — et les **schedules** de learning rate
qui terminent le tableau.

**Objectifs** : comparer les états et les trajectoires, puis étudier les schedules. **Prérequis** : gradients, tableaux et rétropropagation. **Durée** : 2–3 heures. [Version Python](3.2-Optimisateurs.ipynb).


Installer MathNet.Numerics 5.0, TorchSharp-cpu 0.107 et CPython avec scikit-learn. Définir `PYTHONNET_PYDLL`, et si nécessaire `PYTHONHOME` et `PYTHONPATH`. Python.NET sert au vrai jeu de données sklearn.


***

## 1. Le problème visible : le même paysage, trois destins

Le terrain des trajectoires : la fonction de **Beale**, courbée en vallon étroit, minimum global
en $(3, 0{,}5)$. Le rappel de [2.2](../02-ML-Cours/2.2-Descente-de-gradient.ipynb) y prend sa
forme profonde : le learning rate influe sur la trajectoire et la convergence selon le paysage et le budget.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
#r "nuget: TorchSharp-cpu, 0.107.0"
#r "nuget: PythonNet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Globalization;
using System.Net;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.LinearAlgebra.Double;
using TorchSharp;
using static TorchSharp.torch;
using Python.Runtime;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
torch.set_num_threads(1);
if (!PythonEngine.IsInitialized)
{
    var dll = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (string.IsNullOrWhiteSpace(dll))
        throw new Exception("Définir PYTHONNET_PYDLL vers CPython");
    Runtime.PythonDLL = dll;
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}

using (Py.GIL())
{
    dynamic np = Py.Import("numpy"), sk = Py.Import("sklearn"), sys = Py.Import("sys");
    Console.WriteLine($".NET {Environment.Version} | CPython {sys.version} | NumPy {np.__version__} | sklearn {sk.__version__} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version} | TorchSharp {typeof(Tensor).Assembly.GetName().Version}");
}

record PlotSvg(string Svg);
Formatter.Register<PlotSvg>((p, w) => w.Write(p.Svg), "text/html");
string F(double d) => d.ToString("0.###", CultureInfo.InvariantCulture);
string Esc(string s) => WebUtility.HtmlEncode(s);
string StartSvg(int width, int height, string title) => $"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'><rect width='{width}' height='{height}' fill='white'/><text x='{width / 2}' y='22' text-anchor='middle' font-family='sans-serif' font-size='14'>{Esc(title)}</text>";
string Axis(double x, double y, double w, double h, string xlabel, string ylabel, double xmin, double xmax, double ymin, double ymax)
{
    var sb = new StringBuilder();
    sb.Append($"<rect x='{F(x)}' y='{F(y)}' width='{F(w)}' height='{F(h)}' fill='white' stroke='#333'/>");
    for (int k = 0; k <= 4; k++)
    {
        double px = x + w * k / 4, py = y + h - h * k / 4;
        sb.Append($"<text x='{F(px)}' y='{F(y + h + 15)}' text-anchor='middle' font-size='10'>{F(xmin + (xmax - xmin) * k / 4)}</text><text x='{F(x - 6)}' y='{F(py + 3)}' text-anchor='end' font-size='10'>{F(ymin + (ymax - ymin) * k / 4)}</text>");
    }

    sb.Append($"<text x='{F(x + w / 2)}' y='{F(y + h + 31)}' text-anchor='middle' font-size='11'>{Esc(xlabel)}</text><text transform='translate({F(x - 38)} {F(y + h / 2)}) rotate(-90)' text-anchor='middle' font-size='11'>{Esc(ylabel)}</text>");
    return sb.ToString();
}

double Beale(double x, double y) => Math.Pow(1.5 - x + x * y, 2) + Math.Pow(2.25 - x + x * y * y, 2) + Math.Pow(2.625 - x + x * y * y * y, 2);
double[] BealeGrad(double x, double y)
{
    double a = 1.5 - x + x * y, b = 2.25 - x + x * y * y, c = 2.625 - x + x * y * y * y;
    return new[]
    {
        2 * a * (y - 1) + 2 * b * (y * y - 1) + 2 * c * (y * y * y - 1),
        2 * a * x + 2 * b * 2 * x * y + 2 * c * 3 * x * y * y
    };
}

var analytic = BealeGrad(-1.1, .3);
for (int j = 0; j < 2; j++)
{
    double e = 1e-5;
    double numeric = j == 0 ? (Beale(-1.1 + e, .3) - Beale(-1.1 - e, .3)) / (2 * e) : (Beale(-1.1, .3 + e) - Beale(-1.1, .3 - e)) / (2 * e);
    if (Math.Abs(analytic[j] - numeric) > 1e-7)
        throw new Exception("Gradient Beale");
}

Console.WriteLine("Beale : gradient vérifié par différences centrales, minimum (3,0.5)");
var grid = new double[400, 400];
for (int i = 0; i < 400; i++)
    for (int j = 0; j < 400; j++)
        grid[i, j] = Beale(-4.5 + 9.0 * i / 399, -4.5 + 9.0 * j / 399);
string Contours(double px, double py, double width, double height)
{
    var path = new StringBuilder();
    foreach (double level in new[]
    {
        .1,
        .3,
        1.0,
        3.0,
        10.0,
        30.0,
        100.0,
        300.0,
        1000.0
    }

    )
    {
        for (int i = 0; i < 399; i++)
            for (int j = 0; j < 399; j++)
            {
                double v = grid[i, j];
                bool left = (v < level) != (grid[i + 1, j] < level), up = (v < level) != (grid[i, j + 1] < level);
                if (left || up)
                {
                    double x = px + width * (i + .5) / 399, y = py + height - height * (j + .5) / 399;
                    if (left)
                        path.Append($"M{F(x)} {F(y)}h1");
                    if (up)
                        path.Append($"M{F(x)} {F(y)}v1");
                }
            }
    }

    return $"<path d='{path}' fill='none' stroke='#abc9b8' stroke-width='.7'/>";
}

string BealePlot(string title, List<(string, List<double[]>)> paths)
{
    double px = 65, py = 48, w = 430, hh = 330;
    var sb = new StringBuilder(StartSvg(560, 480, title));
    sb.Append(Axis(px, py, w, hh, "x", "y", -4.5, 4.5, -4.5, 4.5));
    sb.Append(Contours(px, py, w, hh));
    string[] colors =
    {
        "#cb4b45",
        "#356cb0",
        "#299965",
        "#9c64aa",
        "#d69122"
    };
    for (int n = 0; n < paths.Count; n++)
    {
        var(label, trajectory) = paths[n];
        var segment = new StringBuilder();
        foreach (var p in trajectory)
        {
            if (p.All(double.IsFinite) && Math.Abs(p[0]) <= 4.5 && Math.Abs(p[1]) <= 4.5)
                segment.Append(F(px + w * (p[0] + 4.5) / 9) + "," + F(py + hh - hh * (p[1] + 4.5) / 9) + " ");
        }

        sb.Append($"<polyline points='{segment}' fill='none' stroke='{colors[n % colors.Length]}' stroke-width='1.6'/><text x='{65 + n * 95}' y='465' font-size='9' fill='{colors[n % colors.Length]}'>{Esc(label)}</text>");
    }

    sb.Append($"<circle cx='{F(px + w * (3 + 4.5) / 9)}' cy='{F(py + hh - hh * (.5 + 4.5) / 9)}' r='4' fill='#222'/><circle cx='{F(px + w * (-1.5 + 4.5) / 9)}' cy='{F(py + hh - hh * (-1.5 + 4.5) / 9)}' r='4' fill='white' stroke='#222'/><text x='75' y='435' font-size='10'>noir : minimum (3, 0.5) ; cercle blanc : départ (-1.5, -1.5)</text></svg>");
    return sb.ToString();
}

string Curves(string title, Dictionary<string, List<(int, double)>> runs)
{
    double px = 70, py = 48, w = 500, hh = 300;
    var all = runs.Values.SelectMany(a => a).Where(p => double.IsFinite(p.Item2) && p.Item2 > 0).ToArray();
    double ymin = Math.Floor(Math.Log10(all.Min(p => p.Item2))), ymax = Math.Ceiling(Math.Log10(all.Max(p => p.Item2)));
    if (ymax <= ymin)
        ymax = ymin + 1;
    int maxit = all.Max(p => p.Item1);
    var sb = new StringBuilder(StartSvg(640, 450, title));
    sb.Append(Axis(px, py, w, hh, "itération", "log10(loss)", 0, maxit, ymin, ymax));
    string[] colors =
    {
        "#cb4b45",
        "#356cb0",
        "#299965",
        "#9c64aa",
        "#d69122"
    };
    int k = 0;
    foreach (var pair in runs)
    {
        string pts = string.Join(" ", pair.Value.Where(p => p.Item2 > 0 && double.IsFinite(p.Item2)).Select(p => F(px + w * p.Item1 / maxit) + "," + F(py + hh - hh * (Math.Log10(p.Item2) - ymin) / (ymax - ymin))));
        sb.Append($"<polyline points='{pts}' fill='none' stroke='{colors[k % 5]}' stroke-width='2'/><text x='{70 + (k % 3) * 175}' y='{399 + (k / 3) * 20}' font-size='10' fill='{colors[k % 5]}'>{Esc(pair.Key)}</text>");
        k++;
    }

    return sb + "</svg>";
}

List<double[]> GDPath(double lr, double momentum)
{
    var w = new[]
    {
        -1.5,
        -1.5
    };
    var buf = new double[2];
    var path = new List<double[]>
    {
        (double[])w.Clone()
    };
    for (int t = 0; t < 400; t++)
    {
        var g = BealeGrad(w[0], w[1]);
        for (int j = 0; j < 2; j++)
        {
            buf[j] = momentum * buf[j] + g[j];
            w[j] -= lr * buf[j];
        }

        if (w.Any(v => !double.IsFinite(v)))
            break;
        path.Add((double[])w.Clone());
    }

    return path;
}

var cases = new[]
{
    ("GD .001", GDPath(.001, 0)),
    ("GD .01", GDPath(.01, 0)),
    ("Momentum .01", GDPath(.01, .9))
};
string Compose3(string a, string b, string c) => "<svg xmlns='http://www.w3.org/2000/svg' width='1680' height='480' viewBox='0 0 1680 480'><rect width='1680' height='480' fill='white'/>" + a.Replace("<svg ", "<svg x='0' y='0' ") + b.Replace("<svg ", "<svg x='560' y='0' ") + c.Replace("<svg ", "<svg x='1120' y='0' ") + "</svg>";
foreach (var(name, path)in cases)
{
    double distance = Math.Sqrt(Math.Pow(path[^1][0] - 3, 2) + Math.Pow(path[^1][1] - .5, 2));
    string final = double.IsFinite(distance) && distance < 1e4 ? $"distance finale {distance:G5}" : "divergence numérique, sortie du cadre";
    Console.WriteLine($"{name} : {path.Count - 1} pas finis, {final}");
}

display(new PlotSvg(Compose3(BealePlot(cases[0].Item1, new() { cases[0] }), BealePlot(cases[1].Item1, new() { cases[1] }), BealePlot(cases[2].Item1, new() { cases[2] }))));


Installed Packages MathNet.Numerics, 5.0.0 PythonNet, 3.0.5 TorchSharp-cpu, 0.107.0

.NET 10.0.14 | CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | NumPy 2.4.4 | sklearn 1.9.1 | Python.NET 3.0.5.0 | TorchSharp 0.107.0.0


Beale : gradient vérifié par différences centrales, minimum (3,0.5)


GD .001 : 400 pas finis, distance finale 1.4976


GD .01 : 400 pas finis, distance finale 0.14238


Momentum .01 : 10 pas finis, divergence numérique, sortie du cadre


GD .001 -4.5 -4.5 -2.25 -2.25 0 0 2.25 2.25 4.5 4.5 x y <path d='M396.391 200.594h1M396.391 199.767h1M397.469 201.421h1M397.469 201.421v1M397.469 199.767v1M397.469 198.94h1M398.546 202.248v1M398.546 198.94v1M399.624 202.248v1M399.624 198.94v1M399.624 198.113h1M400.702 202.248v1M400.702 198.113v1M400.702 197.286h1M401.779 202.248v1M401.779 201.421h1M401.779 197.286v1M402.857 201.421v1M402.857 197.286v1M402.857 196.459h1M403.935 201.421v1M403.935 196.459v1M405.013 201.421v1M405.013 196.459v1M406.09 201.421v1M406.09 196.459v1M406.09 195.632h1M407.168 201.421v1M407.168 200.594h1M407.168 195.632v1M408.246 200.594v1M408.246 195.632v1M409.323 200.594v1M409.323 195.632v1M409.323 194.805h1M410.401 200.594v1M410.401 199.767h1M410.401 194.805v1M411.479 199.767v1M411.479 194.805v1M412.556 199.767v1M412.556 194.805v1M412.556 193.977h1M413.634 199.767v1M413.634 193.977v1M414.712 199.767v1M414.712 198.94h1M414.712 193.977v1M415.789 198.94v1M415.789 193.977v1M416.867 198.94v1M416.867 193.977v1M416.867 193.15h1M417.945 198.94v1M417.945 198.113h1M417.945 193.15v1M419.023 198.113v1M419.023 193.15v1M420.1 198.113v1M420.1 193.15v1M421.178 198.113v1M421.178 197.286h1M421.178 193.15v1M421.178 192.323h1M422.256 197.286v1M422.256 192.323v1M423.333 197.286v1M423.333 192.323v1M424.411 197.286v1M424.411 196.459h1M424.411 192.323v1M425.489 196.459v1M425.489 192.323v1M426.566 196.459v1M426.566 192.323v1M426.566 191.496h1M427.644 196.459v1M427.644 191.496v1M428.722 196.459v1M428.722 195.632h1M428.722 191.496v1M429.799 195.632v1M429.799 191.496v1M430.877 195.632v1M430.877 191.496v1M431.955 195.632v1M431.955 191.496v1M431.955 190.669h1M433.033 195.632v1M433.033 194.805h1M433.033 190.669v1M434.11 194.805v1M434.11 190.669v1M435.188 194.805v1M435.188 190.669v1M436.266 194.805v1M436.266 193.977h1M436.266 190.669v1M437.343 193.977v1M437.343 190.669v1M438.421 193.977v1M438.421 190.669v1M439.499 193.977v1M439.499 190.669v1M439.499 189.842h1M440.576 193.977v1M440.576 189.842v1M441.654 193.977v1M441.654 193.15h1M441.654 189.842v1M442.732 193.15v1M442.732 189.842v1M443.81 193.15v1M443.81 189.842v1M444.887 193.15v1M444.887 189.842v1M445.965 193.15v1M445.965 192.323h1M445.965 189.842v1M447.043 192.323v1M447.043 189.842v1M448.12 192.323v1M448.12 189.842v1M448.12 189.015h1M449.198 192.323v1M449.198 189.015v1M450.276 192.323v1M450.276 189.015v1M451.353 192.323v1M451.353 191.496h1M451.353 189.015v1M452.431 191.496v1M452.431 189.015v1M453.509 191.496v1M453.509 189.015v1M454.586 191.496v1M454.586 189.015v1M455.664 191.496v1M455.664 189.015v1M456.742 191.496v1M456.742 190.669h1M456.742 189.015v1M457.82 190.669v1M457.82 189.015v1M457.82 188.188h1M458.897 190.669v1M458.897 188.188v1M459.975 190.669v1M459.975 188.188v1M461.053 190.669v1M461.053 188.188v1M462.13 190.669v1M462.13 188.188v1M463.208 190.669v1M463.208 189.842h1M463.208 188.188v1M464.286 189.842v1M464.286 188.188v1M465.363 189.842v1M465.363 188.188v1M466.441 189.842v1M466.441 188.188v1M467.519 189.842v1M467.519 188.188v1M468.596 189.842v1M468.596 188.188v1M468.596 187.361h1M469.674 189.842v1M469.674 189.015h1M469.674 187.361v1M470.752 189.015v1M470.752 187.361v1M471.83 189.015v1M471.83 187.361v1M472.907 189.015v1M472.907 187.361v1M473.985 189.015v1M473.985 187.361v1M475.063 189.015v1M475.063 187.361v1M476.14 189.015v1M476.14 187.361v1M477.218 189.015v1M477.218 188.188h1M477.218 187.361v1M478.296 188.188v1M478.296 187.361v1M479.373 188.188v1M479.373 187.361v1M480.451 188.188v1M480.451 187.361v1M481.529 188.188v1M481.529 187.361v1M482.607 188.188v1M482.607 187.361v1M483.684 188.188v1M483.684 187.361v1M484.762 188.188v1M484.762 187.361v1M484.762 186.534h1M485.84 188.188v1M485.84 187.361h1M485.84 186.534v1M486.917 187.361v1M486.917 186.534v1M487.995 187.361v1M487.995 186.534v1M489.073 187.361v1M489.073 186.534v1M490.15 187.361v1M490.15 186.534v1M491.228 187.361v1M491.228 186.534v1M492.306 187.361v1M492.306 186.534h1M492.306 186.534v1M383.459 204.729h1M383.459 203.902h1M383.459 203.075h1M384.536 205.556h1M384.536 205.556v1M3

**Lecture.** Après 400 pas, GD à .001 reste à une distance **1,498** du minimum, tandis que GD à .01 arrive à **0,142**. Avec momentum et .01, la trajectoire quitte le cadre et le calcul devient non fini. Ces observations concernent ce départ et ce budget.


***

## 2. Momentum : la boule dans le ravin

L'idée de Polyak (1964), devenue la norme avec Rumelhart : la mise à jour **accumule** le
gradient au lieu de le consommer brut — $v \leftarrow \mu v + \nabla L$, puis
$\theta \leftarrow \theta - \eta v$ avec $\mu = 0{,}9$. Dans un vallon étroit, les
composantes qui pointent à travers le ravin s'annulent de pas en pas (elles alternent de
signe), les composantes le long du ravin s'additionnent : la bille oscille moins et avance plus
vite. C'est la lecture d'usage du phénomène visible en section 1 : le momentum amplifie le
mouvement cohérent, pas le bruit.

In [2]:
class State
{
    public double[] M, V, Acc, Buf;
    public State(int n)
    {
        M = new double[n];
        V = new double[n];
        Acc = new double[n];
        Buf = new double[n];
    }
}

void Update(double[] p, double[] g, State s, string kind, double lr, int t)
{
    for (int i = 0; i < p.Length; i++)
    {
        double step = g[i];
        if (kind == "Momentum")
        {
            s.Buf[i] = .9 * s.Buf[i] + g[i];
            step = s.Buf[i];
        }
        else if (kind == "Adagrad")
        {
            s.Acc[i] += g[i] * g[i];
            step = g[i] / (Math.Sqrt(s.Acc[i]) + 1e-10);
        }
        else if (kind == "RMSProp")
        {
            s.Acc[i] = .99 * s.Acc[i] + .01 * g[i] * g[i];
            step = g[i] / (Math.Sqrt(s.Acc[i]) + 1e-8);
        }
        else if (kind == "Adam")
        {
            s.M[i] = .9 * s.M[i] + .1 * g[i];
            s.V[i] = .999 * s.V[i] + .001 * g[i] * g[i];
            step = (s.M[i] / (1 - Math.Pow(.9, t))) / (Math.Sqrt(s.V[i] / (1 - Math.Pow(.999, t))) + 1e-8);
        }

        p[i] -= lr * step;
    }
}

// Quatre tenseurs de formes (2,32), (32), (32,2), (2), gradients variables.
double[] RandomNormal(int seed, int n)
{
    var rng = new Random(seed);
    var a = new double[n];
    for (int i = 0; i < n; i++)
        a[i] = Math.Sqrt(-2 * Math.Log(1 - rng.NextDouble())) * Math.Cos(2 * Math.PI * (1 - rng.NextDouble()));
    return a;
}

double[, ] MatrixSlice(double[] a, int start, int rows, int cols)
{
    var m = new double[rows, cols];
    for (int i = 0; i < rows; i++)
        for (int j = 0; j < cols; j++)
            m[i, j] = a[start + i * cols + j];
    return m;
}

Tensor[] ShapeTensors(double[] a) => new[]
{
    torch.tensor(MatrixSlice(a, 0, 2, 32), dtype: ScalarType.Float64),
    torch.tensor(a.Skip(64).Take(32).ToArray(), dtype: ScalarType.Float64),
    torch.tensor(MatrixSlice(a, 96, 32, 2), dtype: ScalarType.Float64),
    torch.tensor(a.Skip(160).Take(2).ToArray(), dtype: ScalarType.Float64)
};
double CheckTorch(string kind)
{
    double lr = kind == "Adagrad" ? 1.0 : .05;
    var initial = RandomNormal(42, 162);
    var vals = (double[])initial.Clone();
    var state = new State(162);
    var raw = ShapeTensors(initial);
    var ps = raw.Select(t => torch.nn.Parameter(t)).ToArray();
    using dynamic opt = kind switch
    {
        "GD" => (object)torch.optim.SGD(ps, learningRate: .05),
        "Momentum" => torch.optim.SGD(ps, learningRate: .05, momentum: .9),
        "Adagrad" => torch.optim.Adagrad(ps, lr: 1.0, eps: 1e-10),
        "RMSProp" => torch.optim.RMSProp(ps, lr: .05, alpha: .99, eps: 1e-8),
        _ => torch.optim.Adam(ps, lr: .05)};
    double worst = 0;
    for (int t = 1; t <= 3; t++)
    {
        var grad = RandomNormal(100 + t, 162);
        Update(vals, grad, state, kind, lr, t);
        var gs = ShapeTensors(grad);
        for (int k = 0; k < 4; k++)
            ps[k].grad = gs[k];
        opt.step();
        int at = 0;
        for (int k = 0; k < 4; k++)
        {
            var data = ps[k].data<double>().ToArray();
            for (int j = 0; j < data.Length; j++)
                worst = Math.Max(worst, Math.Abs(vals[at++] - data[j]));
            ps[k].grad = null;
            gs[k].Dispose();
        }
    }

    foreach (var p in ps)
        p.Dispose();
    foreach (var t in raw)
        t.Dispose();
    if (worst > 1e-12)
        throw new Exception(kind + " divergent");
    return worst;
}

foreach (string name in new[]
{
    "GD",
    "Momentum"
}

)
    Console.WriteLine($"{name} : trois gradients variables, 162 paramètres, écart max {CheckTorch(name):E3}");
// Deux variantes avancées : trois gradients variables sur les vraies API natives.
foreach (string kind in new[]
{
    "Nesterov",
    "AdamW"
}

)
{
    using var p = torch.nn.Parameter(torch.tensor(new double[] { 1, -2 }, dtype: ScalarType.Float64));
    using dynamic opt = kind == "Nesterov" ? (object)torch.optim.SGD(new[] { p }, learningRate: .05, momentum: .9, nesterov: true) : torch.optim.AdamW(new[] { p }, lr: .05, weight_decay: .01);
    double[] expected =
    {
        1,
        -2
    }, buffer =
    {
        0,
        0
    }, m =
    {
        0,
        0
    }, v =
    {
        0,
        0
    };
    var gradients = new[]
    {
        new double[]
        {
            2,
            -3
        },
        new double[]
        {
            -.5,
            1.25
        },
        new double[]
        {
            3,
            .2
        }
    };
    double worst = 0;
    for (int t = 1; t <= 3; t++)
    {
        var g = gradients[t - 1];
        p.grad = torch.tensor(g, dtype: ScalarType.Float64);
        opt.step();
        for (int i = 0; i < 2; i++)
        {
            if (kind == "Nesterov")
            {
                buffer[i] = .9 * buffer[i] + g[i];
                expected[i] -= .05 * (g[i] + .9 * buffer[i]);
            }
            else
            {
                m[i] = .9 * m[i] + .1 * g[i];
                v[i] = .999 * v[i] + .001 * g[i] * g[i];
                expected[i] *= 1 - .05 * .01;
                expected[i] -= .05 * (m[i] / (1 - Math.Pow(.9, t))) / (Math.Sqrt(v[i] / (1 - Math.Pow(.999, t))) + 1e-8);
            }
        }

        worst = Math.Max(worst, expected.Zip(p.data<double>().ToArray(), (a, b) => Math.Abs(a - b)).Max());
        p.grad = null;
    }

    if (worst > 1e-12)
        throw new Exception(kind + " divergent");
    Console.WriteLine($"{kind} : trois gradients variables, écart max {worst:E3}");
}


GD : trois gradients variables, 162 paramètres, écart max 2.220E-016


Momentum : trois gradients variables, 162 paramètres, écart max 2.220E-016


Nesterov : trois gradients variables, écart max 2.220E-016


AdamW : trois gradients variables, écart max 1.110E-016



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

**Lecture.** GD et momentum sont comparés à leurs optimiseurs TorchSharp réels sur quatre tenseurs et trois gradients variables. L’écart maximal affiché mesure ce protocole en float64 ; il ne démontre pas une équivalence pour tous les états possibles.


***

## 3. Adagrad : un learning rate par paramètre

Duchi et al. (2011) renversent la question : au lieu d'un $\eta$ global, chaque paramètre
reçoit un pas divisé par la **racine de la somme des carrés** de ses gradients passés :

$$G \leftarrow G + g \odot g, \quad \theta \leftarrow \theta - \eta \, \frac{g}{\sqrt{G} + \varepsilon}$$

Un paramètre souvent remué (grand gradient fréquent) voit son pas rétrécir ; un paramètre
calme garde un pas presque entier. Sur Beale, cela se voit immédiatement : Adagrad démarre
vite puis **ralentit structurellement** — $G$ ne fait que croître, le dénominateur ne décroît pas, même si le pas effectif dépend aussi du gradient courant.
C'est le défaut célèbre : l'accumulation infinie.

In [3]:
Console.WriteLine($"Adagrad : trois gradients variables, écart max {CheckTorch("Adagrad"):E3}");


Adagrad : trois gradients variables, écart max 4.441E-016


**Lecture.** Adagrad maintient la somme des carrés des gradients ; les trois pas ci-dessus comparent les mêmes états initiaux et hyperparamètres avec TorchSharp. Son dénominateur ne diminue pas, même si le pas effectif dépend du gradient.


***

## 4. RMSProp puis Adam : réparer l'accumulation, ajouter l'élan

**RMSProp** (Hinton, cours 6e, 2012 — jamais publié en article) remplace la somme infinie par
une **moyenne glissante** : $v \leftarrow \alpha v + (1-\alpha) g^2$ avec $\alpha = 0{,}99$.
La moyenne glissante peut adapter le pas aux gradients récents ; elle ne garantit pas un pas stationnaire.

**Adam** (Kingma & Ba, 2014) ajoute la seconde moitié : un momentum $m$ sur le gradient
**et** la normalisation $v$, plus la **correction de biais des premiers pas** — à l'initialisation
$m$ et $v$ partent de zéro, donc $m/(1-\beta_1^t)$ et $v/(1-\beta_2^t)$ ré-échelonnent les
premières itérations pour corriger les estimateurs initialisés à zéro : selon les gradients, le pas non corrigé peut être plus grand ou plus petit :

$$m \leftarrow \beta_1 m + (1-\beta_1) g, \quad v \leftarrow \beta_2 v + (1-\beta_2) g^2$$
$$\theta \leftarrow \theta - \eta \, \frac{m / (1-\beta_1^t)}{\sqrt{v / (1-\beta_2^t)} + \varepsilon}$$

avec $(\beta_1, \beta_2, \varepsilon) = (0{,}9, 0{,}999, 10^{-8})$ — les défauts de torch.

In [4]:
foreach (string name in new[]
{
    "RMSProp",
    "Adam"
}

)
    Console.WriteLine($"{name} : trois gradients variables, écart max {CheckTorch(name):E3}");


RMSProp : trois gradients variables, écart max 8.882E-016


Adam : trois gradients variables, écart max 4.441E-016


**Lecture.** RMSProp et Adam sont comparés à TorchSharp sur trois gradients variables. Les moments Adam sont non nuls dès le premier gradient ; la correction de biais rééchelonne ces estimateurs et ne garantit pas un premier pas plus petit.


***

## 5. Le comparatif 2D : cinq trajectoires sur Beale

Même départ (-1,5 ; -1,5), 400 pas, même fonction — mais le learning rate de référence **par
famille** : c'est la mesure honnête (à lr égal, aucun régime ne se compare : GD exige petit,
Adagrad exige grand). Ce que la superposition doit révéler : la **personnalité du chemin**, pas
seulement le point d'arrivée.

In [5]:
var conf = new[]
{
    ("GD", .01),
    ("Momentum", .005),
    ("Adagrad", 1.0),
    ("RMSProp", .1),
    ("Adam", .1)
};
List<double[]> BealeRun(string name, double lr)
{
    var p = new[]
    {
        -1.5,
        -1.5
    };
    var state = new State(2);
    var path = new List<double[]>
    {
        (double[])p.Clone()
    };
    for (int t = 1; t <= 400; t++)
    {
        Update(p, BealeGrad(p[0], p[1]), state, name, lr, t);
        path.Add((double[])p.Clone());
    }

    return path;
}

var bealeRuns = conf.Select(c => (c.Item1, BealeRun(c.Item1, c.Item2))).ToList();
foreach (var(name, path)in bealeRuns)
{
    double d = Math.Sqrt(Math.Pow(path[^1][0] - 3, 2) + Math.Pow(path[^1][1] - .5, 2));
    Console.WriteLine($"{name, -9} : distance finale {d:F6}");
}

display(new PlotSvg(BealePlot("Beale : cinq trajectoires, 400 pas", bealeRuns.Select((r, i) => ($"{r.Item1} lr={F(conf[i].Item2)}", r.Item2)).ToList())));


GD        : distance finale 0.142378


Momentum  : distance finale 0.000598


Adagrad   : distance finale 0.205361


RMSProp   : distance finale 0.017557


Adam      : distance finale 0.193485


Beale : cinq trajectoires, 400 pas -4.5 -4.5 -2.25 -2.25 0 0 2.25 2.25 4.5 4.5 x y <path d='M396.391 200.594h1M396.391 199.767h1M397.469 201.421h1M397.469 201.421v1M397.469 199.767v1M397.469 198.94h1M398.546 202.248v1M398.546 198.94v1M399.624 202.248v1M399.624 198.94v1M399.624 198.113h1M400.702 202.248v1M400.702 198.113v1M400.702 197.286h1M401.779 202.248v1M401.779 201.421h1M401.779 197.286v1M402.857 201.421v1M402.857 197.286v1M402.857 196.459h1M403.935 201.421v1M403.935 196.459v1M405.013 201.421v1M405.013 196.459v1M406.09 201.421v1M406.09 196.459v1M406.09 195.632h1M407.168 201.421v1M407.168 200.594h1M407.168 195.632v1M408.246 200.594v1M408.246 195.632v1M409.323 200.594v1M409.323 195.632v1M409.323 194.805h1M410.401 200.594v1M410.401 199.767h1M410.401 194.805v1M411.479 199.767v1M411.479 194.805v1M412.556 199.767v1M412.556 194.805v1M412.556 193.977h1M413.634 199.767v1M413.634 193.977v1M414.712 199.767v1M414.712 198.94h1M414.712 193.977v1M415.789 198.94v1M415.789 193.977v1M416.867 198.94v1M416.867 193.977v1M416.867 193.15h1M417.945 198.94v1M417.945 198.113h1M417.945 193.15v1M419.023 198.113v1M419.023 193.15v1M420.1 198.113v1M420.1 193.15v1M421.178 198.113v1M421.178 197.286h1M421.178 193.15v1M421.178 192.323h1M422.256 197.286v1M422.256 192.323v1M423.333 197.286v1M423.333 192.323v1M424.411 197.286v1M424.411 196.459h1M424.411 192.323v1M425.489 196.459v1M425.489 192.323v1M426.566 196.459v1M426.566 192.323v1M426.566 191.496h1M427.644 196.459v1M427.644 191.496v1M428.722 196.459v1M428.722 195.632h1M428.722 191.496v1M429.799 195.632v1M429.799 191.496v1M430.877 195.632v1M430.877 191.496v1M431.955 195.632v1M431.955 191.496v1M431.955 190.669h1M433.033 195.632v1M433.033 194.805h1M433.033 190.669v1M434.11 194.805v1M434.11 190.669v1M435.188 194.805v1M435.188 190.669v1M436.266 194.805v1M436.266 193.977h1M436.266 190.669v1M437.343 193.977v1M437.343 190.669v1M438.421 193.977v1M438.421 190.669v1M439.499 193.977v1M439.499 190.669v1M439.499 189.842h1M440.576 193.977v1M440.576 189.842v1M441.654 193.977v1M441.654 193.15h1M441.654 189.842v1M442.732 193.15v1M442.732 189.842v1M443.81 193.15v1M443.81 189.842v1M444.887 193.15v1M444.887 189.842v1M445.965 193.15v1M445.965 192.323h1M445.965 189.842v1M447.043 192.323v1M447.043 189.842v1M448.12 192.323v1M448.12 189.842v1M448.12 189.015h1M449.198 192.323v1M449.198 189.015v1M450.276 192.323v1M450.276 189.015v1M451.353 192.323v1M451.353 191.496h1M451.353 189.015v1M452.431 191.496v1M452.431 189.015v1M453.509 191.496v1M453.509 189.015v1M454.586 191.496v1M454.586 189.015v1M455.664 191.496v1M455.664 189.015v1M456.742 191.496v1M456.742 190.669h1M456.742 189.015v1M457.82 190.669v1M457.82 189.015v1M457.82 188.188h1M458.897 190.669v1M458.897 188.188v1M459.975 190.669v1M459.975 188.188v1M461.053 190.669v1M461.053 188.188v1M462.13 190.669v1M462.13 188.188v1M463.208 190.669v1M463.208 189.842h1M463.208 188.188v1M464.286 189.842v1M464.286 188.188v1M465.363 189.842v1M465.363 188.188v1M466.441 189.842v1M466.441 188.188v1M467.519 189.842v1M467.519 188.188v1M468.596 189.842v1M468.596 188.188v1M468.596 187.361h1M469.674 189.842v1M469.674 189.015h1M469.674 187.361v1M470.752 189.015v1M470.752 187.361v1M471.83 189.015v1M471.83 187.361v1M472.907 189.015v1M472.907 187.361v1M473.985 189.015v1M473.985 187.361v1M475.063 189.015v1M475.063 187.361v1M476.14 189.015v1M476.14 187.361v1M477.218 189.015v1M477.218 188.188h1M477.218 187.361v1M478.296 188.188v1M478.296 187.361v1M479.373 188.188v1M479.373 187.361v1M480.451 188.188v1M480.451 187.361v1M481.529 188.188v1M481.529 187.361v1M482.607 188.188v1M482.607 187.361v1M483.684 188.188v1M483.684 187.361v1M484.762 188.188v1M484.762 187.361v1M484.762 186.534h1M485.84 188.188v1M485.84 187.361h1M485.84 186.534v1M486.917 187.361v1M486.917 186.534v1M487.995 187.361v1M487.995 186.534v1M489.073 187.361v1M489.073 186.534v1M490.15 187.361v1M490.15 186.534v1M491.228 187.361v1M491.228 186.534v1M492.306 187.361v1M492.306 186.534h1M492.306 186.534v1M383.459 204.729h1M383.459 203.902h1M383.459 203.075h1M384.536 20

**Lecture.** Les distances finales sont **0,142** (GD), **0,0006** (momentum), **0,205** (Adagrad), **0,018** (RMSProp) et **0,193** (Adam) pour les taux indiqués. Momentum atteint ici le voisinage le plus proche, mais les taux diffèrent : ces résultats ne constituent pas un classement général.


***

## 6. Le comparatif MLP : le réseau de 3.1, cinq optimisateurs, trois graines

Le vallon 2D est une métaphore ; le test est le MLP 2-32-2 de [3.1](3.1-Retropropagation.ipynb)
sur `make_moons` (redéfini ci-dessous en compact). Cinq optimisateurs × **trois graines**
{0, 1, 2} (chaque graine = même init de poids pour les cinq), 1500 itérations full-batch,
courbes de perte superposées. Personne n'est favorisé : les lr viennent de la même discipline
de réglage grossier (les plages usuelles par famille).

In [6]:
// Données scikit-learn réelles transférées une fois en C#.
double[][] rows = null;
long[] labels = null;
using (Py.GIL())
{
    dynamic ds = Py.Import("sklearn.datasets");
    using PyObject pair = ds.make_moons(n_samples: 450, noise: .20, random_state: 42);
    using var a = pair.GetItem(0);
    using var b = pair.GetItem(1);
    using var r = a.InvokeMethod("tolist");
    using var y = b.InvokeMethod("tolist");
    rows = r.As<double[][]>();
    labels = y.As<long[]>();
}

var X = DenseMatrix.OfRowArrays(rows);
Console.WriteLine($"make_moons sklearn : {X.RowCount} exemples, 2 features, bruit .20, seed 42");
class Network
{
    public Matrix<double> W1, W2;
    public Vector<double> b1, b2;
}

Network Decode(double[] a) => new Network
{
    W1 = DenseMatrix.OfArray(MatrixSlice(a, 0, 2, 32)),
    b1 = DenseVector.OfArray(a.Skip(64).Take(32).ToArray()),
    W2 = DenseMatrix.OfArray(MatrixSlice(a, 96, 32, 2)),
    b2 = DenseVector.OfArray(a.Skip(160).Take(2).ToArray())
};
double[] Encode(Network p) => p.W1.ToRowMajorArray().Concat(p.b1.ToArray()).Concat(p.W2.ToRowMajorArray()).Concat(p.b2.ToArray()).ToArray();
double[] Init(int seed)
{
    var a = RandomNormal(seed, 128);
    var w = new double[162];
    Array.Copy(a, 0, w, 0, 64);
    for (int i = 0; i < 64; i++)
        w[96 + i] = a[64 + i] * .25;
    return w;
}

(double, double[]) LossGrad(double[] theta, Matrix<double> x, long[] yy)
{
    var p = Decode(theta);
    var z1 = x * p.W1;
    for (int i = 0; i < x.RowCount; i++)
        for (int j = 0; j < 32; j++)
            z1[i, j] += p.b1[j];
    var a = z1.Map(v => Math.Max(0, v));
    var z2 = a * p.W2;
    for (int i = 0; i < x.RowCount; i++)
        for (int j = 0; j < 2; j++)
            z2[i, j] += p.b2[j];
    var dz = DenseMatrix.Create(x.RowCount, 2, 0);
    double loss = 0;
    for (int i = 0; i < x.RowCount; i++)
    {
        double m = Math.Max(z2[i, 0], z2[i, 1]), e0 = Math.Exp(z2[i, 0] - m), e1 = Math.Exp(z2[i, 1] - m), den = e0 + e1;
        loss += Math.Log(den) - (z2[i, (int)yy[i]] - m);
        dz[i, 0] = (e0 / den - (yy[i] == 0 ? 1 : 0)) / x.RowCount;
        dz[i, 1] = (e1 / den - (yy[i] == 1 ? 1 : 0)) / x.RowCount;
    }

    var w2 = a.TransposeThisAndMultiply(dz);
    var b2 = DenseVector.Create(2, j => dz.Column(j).Sum());
    var d1 = dz * p.W2.Transpose();
    for (int i = 0; i < x.RowCount; i++)
        for (int j = 0; j < 32; j++)
            if (z1[i, j] <= 0)
                d1[i, j] = 0;
    var g = new Network
    {
        W1 = x.TransposeThisAndMultiply(d1),
        b1 = DenseVector.Create(32, j => d1.Column(j).Sum()),
        W2 = w2,
        b2 = b2
    };
    return (loss / x.RowCount, Encode(g));
}

var extreme = new double[162];
extreme[160] = 1e20;
extreme[161] = 1e20;
var xOne = DenseMatrix.Create(1, 2, 0);
double shiftLoss = LossGrad(extreme, xOne, new long[] { 0 }).Item1;
using (var z = torch.tensor(new double[, ] { { 1e20, 1e20 } }))
using (var y = torch.tensor(new long[] { 0 }))
using (var ce = torch.nn.functional.cross_entropy(z, y))
{
    if (Math.Abs(shiftLoss - Math.Log(2)) > 1e-15 || Math.Abs(ce.item<double>() - shiftLoss) > 1e-15)
        throw new Exception("CE décalée");
}

extreme[160] = 1e308;
extreme[161] = -1e308;
if (LossGrad(extreme, xOne, new long[] { 0 }).Item1 != 0)
    throw new Exception("CE extrême");
Console.WriteLine($"CE stable : décalage commun {shiftLoss:F12}, classe maximale ±1e308 : 0");
var lrs = new Dictionary<string, double>
{
    {
        "GD",
        .5
    },
    {
        "Momentum",
        .05
    },
    {
        "Adagrad",
        .05
    },
    {
        "RMSProp",
        .005
    },
    {
        "Adam",
        .005
    }
};
Matrix<double> Batch(int[] indices) => DenseMatrix.Create(indices.Length, 2, (i, j) => rows[indices[i]][j]);
List<(int, double)> TrainNet(string kind, int seed, double lr, Func<int, double, double> schedule = null, bool minibatch = false, int iterations = 1500)
{
    var p = Init(seed);
    var state = new State(162);
    var rng = new Random(seed + 1000);
    var hist = new List<(int, double)>();
    for (int t = 1; t <= iterations; t++)
    {
        Matrix<double> bx = X;
        long[] by = labels;
        if (minibatch)
        {
            var idx = Enumerable.Range(0, 32).Select(_ => rng.Next(rows.Length)).ToArray();
            bx = Batch(idx);
            by = idx.Select(i => labels[i]).ToArray();
        }

        var(_, g) = LossGrad(p, bx, by);
        Update(p, g, state, kind, schedule == null ? lr : schedule(t, lr), t);
        if (t % 50 == 0)
            hist.Add((t, LossGrad(p, X, labels).Item1));
    }

    return hist;
}

List<(int, double)> Mean(List<List<(int, double)>> runs) => runs[0].Select((p, i) => (p.Item1, runs.Average(r => r[i].Item2))).ToList();
double Median(IEnumerable<double> xs)
{
    var a = xs.OrderBy(x => x).ToArray();
    return a[a.Length / 2];
}

var results = new Dictionary<string, List<List<(int, double)>>>();
foreach (var conf0 in lrs)
{
    results[conf0.Key] = new();
    foreach (int seed in new[]
    {
        0,
        1,
        2
    }

    )
        results[conf0.Key].Add(TrainNet(conf0.Key, seed, conf0.Value));
    var finals = results[conf0.Key].Select(r => r[^1].Item2);
    Console.WriteLine($"{conf0.Key, -9} trois pertes finales : [{string.Join(", ", finals.Select(F))}], médiane {Median(finals):F6}");
}

foreach (int at in new[]
{
    100,
    300,
    500
}

)
    Console.WriteLine($"Moyennes à {at} : " + string.Join(" | ", results.Select(kv => $"{kv.Key}={kv.Value.Average(r => r.Single(x => x.Item1 == at).Item2):F4}")));
display(new PlotSvg(Curves("MLP : cinq optimiseurs, moyennes de 3 graines", results.ToDictionary(k => $"{k.Key} lr={F(lrs[k.Key])}", k => Mean(k.Value)))));


make_moons sklearn : 450 exemples, 2 features, bruit .20, seed 42


CE stable : décalage commun 0.693147180560, classe maximale ±1e308 : 0


GD        trois pertes finales : [0.069, 0.068, 0.072], médiane 0.068942


Momentum  trois pertes finales : [0.07, 0.071, 0.072], médiane 0.070843


Adagrad   trois pertes finales : [0.063, 0.067, 0.07], médiane 0.066930


RMSProp   trois pertes finales : [0.058, 0.057, 0.055], médiane 0.057133


... (2 outputs left out)

Moyennes à 300 : GD=0.1252 | Momentum=0.1318 | Adagrad=0.0897 | RMSProp=0.0759 | Adam=0.0987


Moyennes à 500 : GD=0.0912 | Momentum=0.0930 | Adagrad=0.0769 | RMSProp=0.0668 | Adam=0.0738


MLP : cinq optimiseurs, moyennes de 3 graines 0 -2 375 -1.5 750 -1 1125 -0.5 1500 0 itération log10(loss) GD lr=0.5 Momentum lr=0.05 Adagrad lr=0.05 RMSProp lr=0.005 Adam lr=0.005


warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



**Lecture.** Les médianes finales de ce run C# sont **0,0689** (GD), **0,0708** (momentum), **0,0669** (Adagrad), **0,0571** (RMSProp) et **0,0650** (Adam). À 100 pas, les pertes moyennes vont de 0,145 à 0,241. Ces résultats décrivent ces taux et graines, sans classement universel ni attribution causale à la correction de biais.


## Exercice 1 : Nesterov, convention PyTorch

Implémentez `NesterovStep` avec `buf = momentum*buf+grad`, puis `p -= lr*(grad+momentum*buf)` aux mêmes coordonnées que TorchSharp. Retournez `true` après mutation ; les fixtures comparent trois gradients variables et une trajectoire Beale de 400 pas. Une interprétation lookahead peut utiliser d’autres coordonnées.


In [7]:
// Exercice 1 : convention Nesterov de torch.optim.SGD.
bool? NesterovStep(double[] p, double[] grad, double[] buffer, double lr = .05, double momentum = .9) => null; // TODO
var nProbe = new[]
{
    1.0,
    -2.0
};
var nState = new double[2];
if (NesterovStep(nProbe, new[] { 2.0, -3.0 }, nState) == null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    try
    {
        var p = new[]
        {
            1.0,
            -2.0
        };
        var b = new double[2];
        using var q = torch.nn.Parameter(torch.tensor(new[] { 1.0, -2.0 }, dtype: ScalarType.Float64));
        using var opt = torch.optim.SGD(new[] { q }, learningRate: .05, momentum: .9, nesterov: true);
        bool ok = true;
        foreach (var g in new[]
        {
            new[]
            {
                2.0,
                -3.0
            },
            new[]
            {
                -.5,
                1.25
            },
            new[]
            {
                3.0,
                .2
            }
        }

        )
        {
            ok &= NesterovStep(p, g, b) == true;
            q.grad = torch.tensor(g, dtype: ScalarType.Float64);
            opt.step();
            var a = q.data<double>().ToArray();
            ok &= a.Zip(p, (u, v) => Math.Abs(u - v)).Max() < 1e-12;
            q.grad = null;
        }

        p = new[]
        {
            -1.5,
            -1.5
        };
        b = new double[2];
        using var t = torch.nn.Parameter(torch.tensor(p, dtype: ScalarType.Float64));
        using var op = torch.optim.SGD(new[] { t }, learningRate: .005, momentum: .9, nesterov: true);
        for (int i = 0; i < 400; i++)
        {
            ok &= NesterovStep(p, BealeGrad(p[0], p[1]), b, .005) == true;
            var v = t.data<double>().ToArray();
            t.grad = torch.tensor(BealeGrad(v[0], v[1]), dtype: ScalarType.Float64);
            op.step();
            t.grad = null;
        }

        ok &= t.data<double>().ToArray().Zip(p, (u, v) => Math.Abs(u - v)).Max() < 1e-9;
        Console.WriteLine(ok ? "Fixtures Nesterov validées" : "Réponse incorrecte");
    }
    catch (Exception)
    {
        Console.WriteLine("Réponse incorrecte");
    }
}


Exercice 1 à compléter



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

***

## 7. Les schedules : le learning rate qui descend avec le temps

Dernier étage, avec une question d'honnêteté : **le schedule paie-t-il toujours ?** Deux
schedules canoniques : **step** ($\eta$ divisé par 2 toutes les 500 itérations) et
**cosinus** (décroissance douce de $\eta$ à $\eta/50$). Deux régimes testés, Adam constant
en témoin dans les deux : (a) notre MLP **full-batch déterministe** ; (b) le même en
**mini-batch 32** — le gradient devient bruité, et c'est le régime pour lequel les schedules
ont été inventés.

In [8]:
double StepSchedule(int t, double lr) => lr * Math.Pow(.5, t / 500);
double CosineSchedule(int t, double lr)
{
    double min = lr / 50;
    return min + .5 * (lr - min) * (1 + Math.Cos(Math.PI * t / 1500));
}

if (Math.Abs(StepSchedule(500, .005) - .0025) > 1e-15 || Math.Abs(CosineSchedule(1500, .005) - .0001) > 1e-15)
    throw new Exception("Bornes schedule");
var fullSchedules = new Dictionary<string, Func<int, double, double>>
{
    {
        "constant",
        (t, lr) => lr
    },
    {
        "step",
        StepSchedule
    },
    {
        "cosine",
        CosineSchedule
    }
};
var fullRuns = new Dictionary<string, List<List<(int, double)>>>();
foreach (var pair in fullSchedules)
{
    fullRuns[pair.Key] = new();
    foreach (int seed in new[]
    {
        0,
        1,
        2
    }

    )
        fullRuns[pair.Key].Add(TrainNet("Adam", seed, .005, pair.Value));
    Console.WriteLine($"Full-batch {pair.Key} : médiane finale {Median(fullRuns[pair.Key].Select(r => r[^1].Item2)):F6}");
}

display(new PlotSvg(Curves("Adam full-batch : trois schedules", fullRuns.ToDictionary(k => k.Key, k => Mean(k.Value)))));


Full-batch constant : médiane finale 0.064987


Full-batch step : médiane finale 0.066970


Full-batch cosine : médiane finale 0.067695


Adam full-batch : trois schedules 0 -2 375 -1.5 750 -1 1125 -0.5 1500 0 itération log10(loss) constant step cosine

**Lecture (full-batch).** Les médianes finales sont **0,0650** pour le taux constant, **0,0670** pour step et **0,0677** pour cosine dans ce run de 1500 pas. Les courbes et ces valeurs dépendent du budget et des taux : ralentir ne garantit pas un gain.


In [9]:
var miniRuns = new Dictionary<string, List<List<(int, double)>>>();
foreach (var pair in fullSchedules)
{
    miniRuns[pair.Key] = new();
    foreach (int seed in new[]
    {
        0,
        1,
        2
    }

    )
        miniRuns[pair.Key].Add(TrainNet("Adam", seed, .02, pair.Value, true));
    Console.WriteLine($"Mini-batch 32 {pair.Key} : médiane finale {Median(miniRuns[pair.Key].Select(r => r[^1].Item2)):F6}");
}

Console.WriteLine("Tirages avec remise, seed+1000 identique entre schedules ; perte mesurée sur les 450 points.");
display(new PlotSvg(Curves("Adam mini-batch 32 : trois schedules", miniRuns.ToDictionary(k => k.Key, k => Mean(k.Value)))));


Mini-batch 32 constant : médiane finale 0.076772


Mini-batch 32 step : médiane finale 0.071330


Mini-batch 32 cosine : médiane finale 0.069098


Tirages avec remise, seed+1000 identique entre schedules ; perte mesurée sur les 450 points.


Adam mini-batch 32 : trois schedules 0 -2 375 -1.5 750 -1 1125 -0.5 1500 0 itération log10(loss) constant step cosine

**Lecture (mini-batch).** Avec tirages appariés par graine, les médianes finales sont **0,0768** (constant), **0,0713** (step) et **0,0691** (cosine). Ces résultats décrivent ce protocole et ne prouvent pas à eux seuls un plancher de bruit ni une règle générale.


## Exercice 2 : AdamW et L2

Implémentez `AdamWStep` en appliquant le decay aux anciens paramètres, puis comparez trois pas à TorchSharp. `ComparerAdamL2AdamW` retourne les deux modèles et huit pertes de base après 400 pas, graine 0, lr=.005 et decay=.01. Adam L2 modifie le gradient ; AdamW découple le decay. Les fixtures rejouent les deux vrais optimiseurs.


In [10]:
// Exercice 2 : AdamW, decay appliqué aux anciens paramètres.
bool? AdamWStep(double[] p, double[] grad, State state, int t, double lr = .05, double weightDecay = .01) => null; // TODO
Dictionary<string, (double[], List<double>)> ComparerAdamL2AdamW() => null; // TODO : seed 0, 400 pas, lr .005, decay .01
var awTest = new[]
{
    1.0,
    -2.0
};
var awState = new State(2);
if (AdamWStep(awTest, new[] { 2.0, -3.0 }, awState, 1) == null)
    Console.WriteLine("Exercice 2 à compléter");
else
{
    try
    {
        var p = new[]
        {
            1.0,
            -2.0
        };
        var state = new State(2);
        using var q = torch.nn.Parameter(torch.tensor(p, dtype: ScalarType.Float64));
        using var opt = torch.optim.AdamW(new[] { q }, lr: .05, weight_decay: .01);
        bool ok = true;
        int t = 0;
        foreach (var g in new[]
        {
            new[]
            {
                2.0,
                -3.0
            },
            new[]
            {
                -.5,
                1.25
            },
            new[]
            {
                3.0,
                .2
            }
        }

        )
        {
            ok &= AdamWStep(p, g, state, ++t) == true;
            q.grad = torch.tensor(g, dtype: ScalarType.Float64);
            opt.step();
            ok &= q.data<double>().ToArray().Zip(p, (u, v) => Math.Abs(u - v)).Max() < 1e-12;
            q.grad = null;
        }

        var result = ComparerAdamL2AdamW();
        ok &= result != null;
        if (result != null)
        {
            ok &= result.Count == 2 && result.ContainsKey("Adam L2") && result.ContainsKey("AdamW");
            foreach (string mode in new[]
            {
                "Adam L2",
                "AdamW"
            }

            )
            {
                var(model, history) = result[mode];
                using var weights = torch.nn.Parameter(torch.tensor(Init(0), dtype: ScalarType.Float64));
                using dynamic trainer = mode == "Adam L2" ? (object)torch.optim.Adam(new[] { weights }, lr: .005, weight_decay: .01) : torch.optim.AdamW(new[] { weights }, lr: .005, weight_decay: .01);
                var expected = new List<double>();
                for (int i = 1; i <= 400; i++)
                {
                    var current = weights.data<double>().ToArray();
                    var(_, gradient) = LossGrad(current, X, labels);
                    weights.grad = torch.tensor(gradient, dtype: ScalarType.Float64);
                    trainer.step();
                    weights.grad = null;
                    if (i % 50 == 0)
                        expected.Add(LossGrad(weights.data<double>().ToArray(), X, labels).Item1);
                }

                ok &= model.Length == 162 && model.All(double.IsFinite) && history.Count == 8 && history.All(double.IsFinite) && history.Zip(expected, (a, b) => Math.Abs(a - b)).Max() < 1e-8 && model.Zip(weights.data<double>().ToArray(), (a, b) => Math.Abs(a - b)).Max() < 1e-8;
            }
        }

        Console.WriteLine(ok ? "Fixtures AdamW validées" : "Réponse incorrecte");
    }
    catch (Exception)
    {
        Console.WriteLine("Réponse incorrecte");
    }
}


Exercice 2 à compléter



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

***

## 8. Le pont : retrouver ces optimisateurs dans les autres séries

Ce notebook a ouvert les cinq boîtes ; partout ailleurs du dépôt, elles sont refermées mais
vous savez maintenant ce qu'elles contiennent :

- **[RL](../../../RL/README.md) et PostTraining** : `torch.optim.Adam(model.parameters(), lr=...)`
  — les $(\beta_1, \beta_2, \varepsilon)$ par défaut de ce notebook, la correction de biais
  de la section 4 tournant à chaque step ;
- **ML-Training-Pipeline** : le lr schedule cosine de la section 7 est celui des entraînements
  réels (souvent avec warmup — exercice 3) ;
- **[3.4 (transformer)](README.md)** : le mini-GPT s'entraînera sous AdamW — l'exercice 2 vous
  l'aura déjà écrit.

La règle de lecture : quand une courbe d'entraînement fait quelque chose d'étrange dans les
100 premières itérations, examiner l’échelle du pas, les moments et le schedule, sans en déduire à eux seuls une causalité.

## Exercice 3 : warmup linéaire

`Warmup` calcule `lr * min(1,t/200)`. `ComparerWarmup` retourne pour chaque graine 0,1,2 les bras `froid` et `warmup`, avec paramètres et huit pertes full-data sur 400 pas mini-batch 32. Les tirages `seed+1000` avec remise sont appariés et les fixtures rejouent les deux bras avec TorchSharp.


In [11]:
// Exercice 3 : warmup linéaire et deux bras mini-batch appariés.
double? Warmup(int t, double lr) => null; // TODO
Dictionary<int, Dictionary<string, (double[], List<double>)>> ComparerWarmup() => null; // TODO
if (Warmup(100, .02) == null)
    Console.WriteLine("Exercice 3 à compléter");
else
{
    try
    {
        bool ok = true;
        foreach (double lr in new[]
        {
            .02,
            .1
        }

        )
            foreach (int t in new[]
            {
                0,
                1,
                100,
                200,
                400
            }

            )
                ok &= Warmup(t, lr).HasValue && Math.Abs(Warmup(t, lr).Value - lr * Math.Min(1.0, t / 200.0)) < 1e-12;
        var result = ComparerWarmup();
        ok &= result != null;
        if (result != null)
        {
            ok &= result.Count == 3 && new[]
            {
                0,
                1,
                2
            }.All(result.ContainsKey);
            foreach (int seed in new[]
            {
                0,
                1,
                2
            }

            )
            {
                var arms = result[seed];
                ok &= arms.Count == 2 && arms.ContainsKey("froid") && arms.ContainsKey("warmup");
                foreach (string mode in new[]
                {
                    "froid",
                    "warmup"
                }

                )
                {
                    var(model, history) = arms[mode];
                    using var q = torch.nn.Parameter(torch.tensor(Init(seed), dtype: ScalarType.Float64));
                    using var opt = torch.optim.Adam(new[] { q }, lr: .02);
                    var rng = new Random(seed + 1000);
                    var expected = new List<double>();
                    for (int t = 1; t <= 400; t++)
                    {
                        var idx = Enumerable.Range(0, 32).Select(_ => rng.Next(rows.Length)).ToArray();
                        var(_, g) = LossGrad(q.data<double>().ToArray(), Batch(idx), idx.Select(i => labels[i]).ToArray());
                        q.grad = torch.tensor(g, dtype: ScalarType.Float64);
                        opt.ParamGroups.First().LearningRate = .02 * (mode == "warmup" ? Math.Min(1.0, t / 200.0) : 1.0);
                        opt.step();
                        q.grad = null;
                        if (t % 50 == 0)
                            expected.Add(LossGrad(q.data<double>().ToArray(), X, labels).Item1);
                    }

                    ok &= model.Length == 162 && model.All(double.IsFinite) && history.Count == 8 && history.All(double.IsFinite) && history.Zip(expected, (a, b) => Math.Abs(a - b)).Max() < 1e-8 && model.Zip(q.data<double>().ToArray(), (a, b) => Math.Abs(a - b)).Max() < 1e-8;
                }
            }
        }

        Console.WriteLine(ok ? "Fixtures warmup validées" : "Réponse incorrecte");
    }
    catch (Exception)
    {
        Console.WriteLine("Réponse incorrecte");
    }
}


Exercice 3 à compléter



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## Contre-sens usuels sur les optimiseurs

- Adam calcule des moments non nuls dès le premier gradient ; la correction de biais ne garantit pas un premier pas plus petit.
- La convention Nesterov dépend des coordonnées ; les fixtures utilisent celle de TorchSharp.
- AdamW découple le decay, contrairement à L2 couplé dans le gradient.
- Les classements et effets des schedules dépendent des données, des taux et du budget ; une fixture finie ne prouve pas une supériorité générale.


## À retenir

- **GD** suit le gradient courant avec un taux global ; **momentum** accumule un état qui influence les pas suivants.
- **Adagrad** accumule les carrés des gradients ; **RMSProp** utilise une moyenne glissante ; **Adam** combine deux moments et corrige leur biais d’initialisation.
- Les cinq règles ont été comparées à leurs API réelles sur plusieurs gradients, puis mesurées sur Beale et sur un MLP. Les différences dépendent des taux, données et budgets.
- Les schedules constant, step et cosine sont comparés en full-batch puis sur des mini-batches appariés. Les exercices prolongent Nesterov, AdamW et warmup.

Poursuivez avec [3.3 — Régularisation](README.md).

## References

1. Polyak, B. T. (1964). *Some methods of speeding up the convergence of iteration methods*. USSR Computational Mathematics and Mathematical Physics 4(5):1-17.
2. Duchi, J., Hazan, E., & Singer, Y. (2011). *Adaptive subgradient methods for online learning and stochastic optimization*. JMLR 12:2121-2159.
3. Tieleman, T., & Hinton, G. (2012). *Lecture 6.5 — RMSProp*. Coursera: Neural Networks for Machine Learning.
4. Kingma, D. P., & Ba, J. (2014). *Adam: A method for stochastic optimization*. arXiv:1412.6980.
5. Loshchilov, I., & Hutter, F. (2019). *Decoupled weight decay regularization (AdamW)*. ICLR.

## Vérifiez votre compréhension

1. Que conserve momentum entre deux pas ?
2. Que corrige la correction de biais d’Adam ?
3. Quelle différence entre L2 couplé et AdamW ?
4. Pourquoi un schedule n’est-il pas toujours gagnant ?

<details><summary>Réponses</summary>

1. Un accumulateur de gradients. 2. L’initialisation à zéro des estimateurs de moments ; le premier pas non corrigé n’est pas forcément petit. 3. L2 modifie le gradient, AdamW applique le decay aux anciens paramètres. 4. Son effet dépend du taux, du bruit et du budget.

</details>


## Pour aller plus loin

- [PyTorch : optimiseurs](https://docs.pytorch.org/docs/stable/optim.html).
- [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html).
- [Références du carnet](#references).
